## Using Ollama - The Basics
Ollama is a simple way of accessing open-source Large Language Models (LLMs) and benefitting from LLM functionality with limited (or managed) risk of exposing your data to servers outside of the ONS estate.

Disclaimer: Ollama is one of many tools that achieve similar functionality.

Whether you want to use Ollama from the command line or programatically - e.g. using the Langchain framework - you will need to download Ollama locally.

The download process is simple; it can be accessed [here]("https://ollama.com/"). Downloads are in 3 different flavours: MacOS, Linux or Windows (in beta). Don't expect Ollama to be running on an on-network ONS laptop! However, I have successfully tested the Linux version using WSL2 (before the launch of the Windows version) on my personal Windows 11 computer with very similar results to MacOS.

<img src="images/ollama_website.png" width="50%"/>

Once installed, you do not need to activate the service. It will run automatically (from startup) in the background with the Ollama icon <img src="images/ollama_icon.png" height=15px/> appearing in your MacOS menu bar. For some, this will be unnecessary use of memory and the auto-start can be de-selected in the installation process. However, it's worth noting that an 'idle Ollama' uses a fraction of the memory used by Teams (for example).

You can browse the various LLMs available through Ollama from the **Models** menu. Some of the ones that I have previously tested so far:

<img src="images/gemma.png" width=50%/>
<img src="images/mistral.png" width=50%/>
<img src="images/openhermes.png" width=50%/>
<img src="images/llama2.png" width=50%/>
<img src="images/codellama.png" width=50%/>
<img src="images/llava.png" width=50%/>

Although we won't be looking at LLMs in isolation in this session, it is always worth remembering that each model is trained differently and they have different intended purposes (and pitfalls). A model will always necessarily limited by the data it was trained on and, consequently, you will find models either hallucinating or refusing. Examples of this might include today's date, the name of the current prime minister, etc.

### Using Ollama from the MacOS Terminal
Ollama commands will work straight away from the Terminal without any configuration. Here is a summary of the key commands:

<img src="images/ollama_commands.png"/>

Once you have chosen the LLM that you want to start with, you must download it locally. The model will be available to use from the CLI or programatically after this.

For example, if you want to download the latest **gemma** model:

```bash
$ ollama pull gemma
```

Once installed, you can instantiate a chat instance immmediately. For example:

```bash
$ ollama run gemma
```

<img src="images/ollama_prompt.png"/>

**Note:** if you use the `ollama run` command to launch a model that does not yet exist locally, installation will be triggered automatically and the chat instance will launch on completion.

We will briefly cover the strengths and weaknesses of a selection of open-source models later. However, let's dive in to the very obvious 'wins':

#### Applications
Imagine a brief school report e.g.

*Although Josh has made some good progress in Maths this year, he can be easily distracted in class. This has limited his achievements in English, particularly in Writing tasks. When he focuses, he is capable of explaining his processes to others and he generally uses his sense of humour appropriately. I was delighted to see that Josh was playing in the school football team once again. Next year, he has the potential to be highly successful but he will need to concentrate hard and be willing to put in the extra effort required.*

Using the **gemma** model, built on the same framework as the Gemini Pro models, we can interrogate this in a number of views:

- restatement (reinterpret in a preferred style or format)<br>
&ensp;&ensp;e.g. **Rewrite the following school report in language appropriate for a young child: *add the child's school report here*.**<br><br>
- summarisation (capture key information, in varying lengths)<br>
&ensp;&ensp;e.g. **Please summarise the following school report in no more than 2 sentences: *add the child's school report here*.**<br><br>
- classification (e.g. grade, sentiment, ...)<br>
&ensp;&ensp;e.g. **What grade from 1 to 5 (1 lowest, 5 highest) would you give the child for effort, given the following school report: *add the child's school report here*.**

#### Limitations
You will note that the section on use from the CLI is very short in comparison to the programmatic application of Ollama. Although fun - and no doubt of interest if chatbots are your thing - there are a number of obvious drawbacks:
1. For most models, it is awkward or impossible to have them accessing files in a storage system. For example, text generally has to be provided as part of your message. An exception to this is the `llava` model.......
2. It is not possible to amend the parameters of an installed model. For a number of use cases in our team, we have set the **temperature** of the model to 0 to limit creativity and hallucination. A new model must be created locally to pass your own model parameter values (details below).
3. Although the CLI is great for ad-hoc use, it severely limits reproducibility and implementation as part of a workflow. Instructions on using Ollama programmatically have a dedicated section below!

#### Bespoke models

Ultimately, new models are built from a `Modelfile`. In the following simple(!) example, we address the **temperature** issue above by drawing on an existing model, passing some bespoke parameters, and storing as a new model. More details about all the parameters that can be set can be found [here]("https://github.com/ollama/ollama/blob/main/docs/modelfile.md#parameter").

<img src="images/modelfile.png"/>

```bash
$ ollama create miss-t-ral -f ./Modelfile
$ ollama run miss-t-ral
```

In order to break a chat instance and return to the Terminal Prompt, simply enter the command `/bye`.

### Using Ollama models programmatically

#### ChatOllama
Let's take a quick look at the like-for-like CLI functionality in Python code. `langchain` helpfully provides multiple wrappers for Ollama. In this first example, we will use `ChatOllama`:

In [4]:
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# instantiate LLM object; model available locally
llm = ChatOllama(model="mistral", temperature=0)

# input to LLM
prompt = ChatPromptTemplate.from_template("Rewrite the following school report in language appropriate for a young child: {report}")

# new LangChain Expressive Language chain syntax (LCEL)
chain = prompt | llm | StrOutputParser()

input = {"report": """Although Josh has made some good progress in Maths this year,
         he can be easily distracted in class. This has limited his achievements in English,
         particularly in Writing tasks. When he focuses, he is capable of explaining his
         processes to others and he generally uses his sense of humour appropriately.
         I was delighted to see that Josh was playing in the school football team once again.
         Next year, he has the potential to be highly successful but he will need to
         concentrate hard and be willing to put in the extra effort required."""}

In [5]:
chain.invoke(input)

" Josh is learning new math numbers and doing a good job! But sometimes, he gets distracted during lessons. This makes it harder for him to write stories in English class. When Josh pays attention, he can share how he solves problems and uses jokes wisely. I'm happy Josh joined the school football team again! Next year, Josh can be a star if he focuses more and works harder."

#### RAG Question Answering

In [20]:
from langchain_community.document_loaders import PyPDFLoader
from langchain import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
fp = "ollama_basics/data/STA198217e_2019_ks2_mathematics_Paper2_reasoning.pdf"
loader = PyPDFLoader(fp)
pages = loader.load_and_split()
pages[3].page_content

'Page 4 of 24 1  In this grid, there are four multiplications.\nWrite the three missing numbers.\n4 × 8 =\n× ×\n3 × = 21\n= =\n563C6  Multiplication grid\n1 mark\nKS2 item template version 2M005807  –  19 October 2018 3:46 PM – Version 5\nWhat number is 1,000 less than 9,072? 24N2b  Thousand less\n1 mark\nKS2 item template version 2M005799  –  19 October 2018 3:43 PM – Version 1\nH00070A0424'

In [31]:
#question = "What is Experimentation?"
prompt_template = """Answer the questions as precisely as possible using the provided context. If the answer is
                    not contained in the context, say "answer not available in context" \n\n
                    Context: \n {context}?\n
                    Answer:
                  """

prompt = PromptTemplate(
    template=prompt_template, input_variables=["context"]
)

In [79]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
embeddings = OllamaEmbeddings()
llm = Ollama(model="mistral", temperature=0)
# context = "\n".join(str(p.page_content) for p in pages[:7])
context = pages[3].page_content
stuff_chain = load_qa_chain(llm, chain_type="stuff", prompt=prompt)
stuff_answer = stuff_chain(
    pages[21:22], return_only_outputs=True
)


In [80]:
stuff_answer

{'output_text': " To find the difference in the number of cubes between Amina's and Stefan's cuboids, we first need to calculate the volume of each cuboid and then subtract the smaller volume from the larger one.\n\nAmina's cuboid dimensions are given as 6 cm x 3 cm x 4 cm. So, its volume is:\n\nVolume_Amina = length × width × height\n                   = 6 cm × 3 cm × 4 cm\n                   = 144 cubic centimeters\n\nStefan's cuboid is described as being 5 cm longer, 5 cm taller, and 5 cm wider than Amina's. So, its dimensions are:\n\nLength_Stefan = 6 cm + 5 cm = 11 cm\nWidth_Stefan = 3 cm + 5 cm = 8 cm\nHeight_Stefan = 4 cm + 5 cm = 9 cm\n\nNow we can calculate the volume of Stefan's cuboid:\n\nVolume_Stefan = length × width × height\n                    = 11 cm × 8 cm × 9 cm\n                    = 992 cubic centimeters\n\nTo find the difference in the number of cubes, we first need to convert both volumes into the same number of cubes. Since each cube is 1 cubic centimeter, we ca

In [6]:
from langchain.document_loaders import PyPDFLoader
loader = PyPDFLoader("ollama_basics/data/Command-A-Crew-Of-AI-Agents.pdf")
documents = loader.load()

In [7]:
from langchain.chains import RetrievalQA
from langchain.indexes import VectorstoreIndexCreator
from langchain.text_splitter import CharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma

# split the documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.llms import Ollama
embeddings = OllamaEmbeddings()
db = Chroma.from_documents(texts, embeddings)

In [49]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k":3})
llm = Ollama(model="llama2", temperature=0)
qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

In [50]:
question = "What is CrewAI?"
output = qa.invoke(question)

In [51]:
output["result"]

'Based on the provided context, CrewAI appears to be a tool or platform that enables users to create customizable agent teams for solving problems. It allows users to allocate tasks to agents and models, and provides a way to dynamically allocate tasks to achieve a specific goal. CrewAI is being actively developed, with plans to introduce consensual (task-switching) and hierarchical (task-prioritization) processing in the future.\n\nIn the provided context, CrewAI is used to create an agent team responsible for summarizing a webpage. The team consists\u200d Хронологија\xa0\u200d\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0\xa0 Unterscheidung between different types of AI models and their respective tasks.\n\nIn the example provided, CrewAI is used to create an agent team for setting up CrewAI in a Python environment. The team consists of two agents: one responsible for researching the topic, and the other for writing instructions for beginners. Th

In [45]:
output["source_documents"]

[Document(page_content='Note that every agent must have a role, goal and backstory. These are passed to the prompt,\ninformation which is interpreted by the LLM.\nFinally, we define the tasks and combine the agents into a Crew! In essence, this packages the\nagents and tasks.\nThe openhermes LLM appeared considerably faster at completing the tasks than mistral in this example\nbut performance is affected by LLM suitability to tasks and the wording in the prompts.\nAgent Tools\nTools can be made available to the agents to perform specific actions that will help them fulfil their\nrole and complete their task(s).\nAlthough there are a number of ‘off-the-shelfʼ tools (some can be found here), we can also create\ncustom tools from which an agent can choose to help it undertake tasks. The langchain@tool\ndecorator is used to introduce each tool. In our next example, we will want to provide an agent with a\ntool to scrape one of the Data Science Campus blog posts:task1 = Task(description=\'W

### Evaluating models

Notes - access GPU. Turn off GPU.  <---- parameters

Other tools worth exploring include [CrewAI]("https://www.crewai.io/") and [PrivateGPT]("https://docs.privategpt.dev/overview/welcome/introduction").